In [1]:
import os
import sys
import subprocess
import pandas as pd
from IPython.display import display

WORDS_DIR = "./data/words/asca"
#OUTPUT_DIR = "./data/rules/applied/asca"
RULES_DIR = "./data/rules/asca"

word_files = []
rule_files = []

for file in os.listdir(WORDS_DIR):
    if file.endswith(".wsca"):
        word_files.append(f"{file}")
print(word_files[1])

weirdness_0.5.wsca


In [2]:
sections_df = pd.read_csv("./data/index_diachronica_sections.csv", dtype={"index": str, "name": str, "rule_count": int})

rules_df = sections_df[sections_df["rule_count"] > 0].copy()
rules_df['rule_file'] = rules_df['index'].apply(lambda x: f"{x}.rsca")

display(rules_df.head(3))

rule_files = rules_df['rule_file'].tolist()


,index,name,rule_count,rule_file
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,11,6.1.rsca
2,6.1.1,Proto-Omotic to North Omotic,18,6.1.1.rsca
3,6.1.1.1,North Omotic to Bench,9,6.1.1.1.rsca


In [3]:
# run asca-rust to validate rules

results = []
word_file = word_files[1] # weirdness 0.5

cwd = os.getcwd()

def run_asca(word_file, rule_file):
    output_file = word_file.replace(".wsca", "") + "_" + rule_file.replace(".rsca", ".wsca")
    #asca_cmd = f"yes | ~/.cargo/bin/asca run {WORDS_DIR}/{word_file} -r {RULES_DIR}/{rule_file} -o {OUTPUT_DIR}/{output_file}"
    asca_cmd = f"~/.cargo/bin/asca run {WORDS_DIR}/{word_file} -r {RULES_DIR}/{rule_file}"

    result = {"rule": rule_file, "returncode": 0, "error": ""}

    try:
        output = subprocess.check_output(asca_cmd, stderr=subprocess.STDOUT, timeout=10, shell=True, universal_newlines=True)
    except subprocess.CalledProcessError as exc:
        result["returncode"] = exc.returncode 
        result["error"] = exc.output.replace("\n", "\\n")
    except subprocess.TimeoutExpired as exc:
        result["returncode"] = 124
        result["error"] = exc.output.decode("utf-8").replace("\n", "\\n")

    return result

for rule_file in rule_files:
    result = run_asca(word_file, rule_file)
    results.append(result)

results_df = pd.DataFrame(results)

results_df.to_csv("./data/asca_results.csv", index=False)

results_df[results_df["returncode"] != 0].head()

,rule,returncode,error
